# 🏢 Automated Company Brochure Generator (Local AI Pipeline)

> 💡 **Overview:** An automated end-to-end pipeline that scrapes a target company website, utilizes a locally hosted LLM (**Llama 3.2** via **Ollama**) to filter high-signal pages, cleans HTML noise, and synthesizes a structured marketing brochure.

```text
[Target URL] ──► (Scrape Links) ──► (LLM Page Filter) ──► (Clean HTML) ──► (LLM Brochure Generation)

### 1. Setup & Imports
Import the required libraries for web scraping (`requests`, `BeautifulSoup`) and LLM orchestration (`openai` pointing to local Ollama).

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
MODEL = "llama3.2"

### 2. Link Extraction & Intelligent Filtering
First, scrape all hyperlinked paths from the base URL. Then, pass the list to the LLM with a strict JSON schema to identify the top 3 pages relevant to company details, mission, or products.

In [13]:
url = "https://huggingface.co"

def fetch_website_links(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    links = [urljoin(url, a['href']) for a in soup.find_all('a', href = True)]
    return list(set(links))

In [14]:
link_system_prompt = """
You are an expert filter. You are given a list of links extracted from a website.
Your task is to select up to 3 links from the provided list that describe the person, company, or curriculum/projects.

Rules:
1. You MUST pick links that exist word-for-word in the user's provided list.
2. NEVER guess, modify, or make up any URL.
3. If no extra pages are relevant, return an empty list.

Output ONLY a JSON object with this exact schema:
{"links": [{"type": "description of page", "url": "exact URL from the list"}]}
"""

In [15]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}...")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}  
    )
    
    result = response.choices[0].message.content
    print("\n#####################\nWithout json.loads",result)
    links = json.loads(result)
    print("\n#####################\nWith json.loads",links)
    print(f"Found {len(links['links'])} relevant links")
    return links

# test = select_relevant_links(url)


  

### 3. Web Scraping & Content Cleaning
Scrape the selected target pages and strip boilerplate clutter (`<nav>`, `<footer>`, `<script>`, `<style>`) to leave clean, token-efficient text for the model.

In [17]:
def extract_page_content(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["footer", "nav", "style", "script"]):
        tag.decompose()

    return soup.get_text(separator=" ", strip=True)

### 4. End-to-End Pipeline & Brochure Generation
Define the brochure prompting templates and build the main function that coordinates link selection, content scraping, and final LLM synthesis.

In [18]:
brochure_system_prompt = """
You are an expert marketing copywriter.
Your task is to synthesize the provided extracted website content into a polished, comprehensive company brochure in Markdown format.

Required Sections:
- ## Company Overview & Mission
- ## Key Offerings, Products & Solutions
- ## Community, Culture & Highlights
- ## How to Get Started / Call to Action

Critical Rules:
1. Strict Grounding: Rely ONLY on facts stated in the provided text. Never invent or assume details (such as founders, founding dates, or unmentioned tools). If a detail is absent, omit it.
2. Distinct Bullets: Ensure all listed products and services are distinct and non-overlapping.
3. Tone: Informative, engaging, and professional.
"""

def get_brochure_user_prompt(company_name, content):
    return f"""
Please generate a comprehensive brochure for '{company_name}' based on the following extracted website content:

{content}
"""

In [25]:
def generate_brochure(company_name, url):
    print("=" * 60)
    print(f"🚀 Pipeline Started: {company_name}")
    print(f"🌐 Target URL: {url}")
    print("=" * 60)
    
    relevant_links = select_relevant_links(url)
    
    all_details = ""
    for item in relevant_links.get("links", []):
        page_url = item.get("url")
        
        if not page_url or not isinstance(page_url, str) or not page_url.startswith("http"):
            continue
            
        print(f"  🔍 [Scraping] -> {page_url}")
        page_text = extract_page_content(page_url)
        all_details += f"\n\n=== Content from {page_url} ===\n\n"
        all_details += page_text
        
    print("-" * 60)
    print(f"📊 Extraction Complete: {len(all_details):,} clean characters collected.")
    print(f"🤖 Synthesizing brochure using {MODEL} (temperature=0.2)...")
    print("-" * 60)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, all_details)}
        ],
        temperature=0.2
    )
    
    brochure_markdown = response.choices[0].message.content
    
    formatted_output = f"""
<div style="border-left: 4px solid #3B82F6; padding: 15px; margin: 10px 0;">

{brochure_markdown}

</div>
"""
    display(Markdown(formatted_output))
    return brochure_markdown

### 5. Run & Display Result
Execute the pipeline on an example target website.

In [ ]:
brochure = generate_brochure("Hugging Face", "https://huggingface.co")